# Qwen vs Gemma Benchmark for Chinese Poem Understanding on Apple Silicon MLX

This notebook benchmarks **inference quality and speed** for Qwen and Gemma-style instruction models on an Apple Silicon Mac, using **MLX / mlx-lm**.

Target machine: **M4 Mac mini, 32 GB unified memory**.

The benchmark is designed for your task: classical Chinese poem or line → modern Chinese understanding / restatement.

It measures:

- output text quality against reference answers
- latency per example
- tokens per second
- approximate first-token latency when `stream_generate` is available
- MLX active/peak memory when exposed by your installed MLX version
- side-by-side qualitative examples
- CSV exports for manual review

> Recommendation: run this notebook in a fresh environment instead of a Paddle/PaddleX environment. This notebook pins NumPy below 2.4 to avoid the PaddleX conflict you saw.

## 0. Environment setup

Run this once in a fresh virtual environment or Conda environment.

Suggested terminal setup:

```bash
python3 -m venv poem-mlx-env
source poem-mlx-env/bin/activate
python -m pip install -U pip
python -m pip install ipykernel
python -m ipykernel install --user --name poem-mlx-env --display-name "poem-mlx-env"
```

Then select the `poem-mlx-env` kernel in Jupyter.

In [ ]:
# Install packages.
# numpy<2.4 avoids conflicts with packages such as paddlex that require numpy<2.4.
# If you are already in a clean environment, this is safe.

%pip install -U "numpy<2.4"     mlx mlx-lm     pandas tqdm matplotlib     transformers sentencepiece protobuf     evaluate rouge-score sacrebleu bert-score     openpyxl

## 1. Imports and system check

In [ ]:
import os
import re
import gc
import json
import time
import platform
import subprocess
from pathlib import Path
from typing import Dict, List, Any, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import matplotlib.pyplot as plt

import mlx.core as mx
from mlx_lm import load, generate

try:
    from mlx_lm import stream_generate
    HAS_STREAM_GENERATE = True
except Exception:
    stream_generate = None
    HAS_STREAM_GENERATE = False

print("Python:", platform.python_version())
print("Platform:", platform.platform())
print("Processor:", platform.processor())
print("MLX stream_generate available:", HAS_STREAM_GENERATE)

try:
    print("MLX default device:", mx.default_device())
except Exception as e:
    print("Could not inspect MLX device:", e)

try:
    out = subprocess.check_output(["sysctl", "-n", "hw.memsize"]).decode().strip()
    print("System memory GB:", round(int(out) / 1024**3, 2))
except Exception:
    pass

## 2. Configuration

The default model IDs are editable. For your Mac mini, start with 4-bit MLX models.

Good first-pass candidates:

- `mlx-community/Qwen3-8B-4bit` or another Qwen 7B/8B Instruct MLX quantization
- `mlx-community/gemma-3-12b-it-4bit` or your chosen Gemma 4/Gemma-family 12B IT MLX quantization

If a repo name fails, search Hugging Face for the current MLX-community quantized repo name and replace it below.

In [ ]:
# -------------------------
# Edit these settings
# -------------------------

MODELS = {
    "qwen_8b_4bit": "mlx-community/Qwen3-8B-4bit",
    "gemma_12b_4bit": "mlx-community/gemma-3-12b-it-4bit",
}

# For a 32GB Mac, keep benchmark size small at first.
MAX_EXAMPLES = 30
RANDOM_SEED = 42

# Generation settings: deterministic for fair comparison.
MAX_TOKENS = 256
TEMPERATURE = 0.0
TOP_P = 1.0

# Optional: reduce KV cache if memory is tight. None lets mlx-lm decide.
MAX_KV_SIZE = None  # e.g. 4096

# Data path. Leave None to auto-detect or use built-in toy examples.
DATA_PATH = None

# Output directory
OUT_DIR = Path("benchmark_outputs")
OUT_DIR.mkdir(exist_ok=True)

## 3. Load benchmark data

Expected columns, in order of preference:

- input column: `input`, `source`, `sentence`, `poem`, `古诗`, `原文`, `text`
- reference column: `target`, `reference`, `answer`, `modern`, `translation`, `现代文`, `译文`

Supported file types: `.csv`, `.xlsx`, `.jsonl`, `.json`.

If no data file is found, the notebook uses a tiny built-in sample just to verify the pipeline.

In [ ]:
def find_data_file() -> Optional[Path]:
    if DATA_PATH:
        p = Path(DATA_PATH)
        return p if p.exists() else None
    candidates = []
    for pattern in ["*.csv", "*.xlsx", "*.jsonl", "*.json"]:
        candidates.extend(Path(".").glob(pattern))
        candidates.extend(Path("/mnt/data").glob(pattern))
    # Avoid using prior benchmark outputs as inputs.
    candidates = [p for p in candidates if "benchmark" not in p.name.lower()]
    return candidates[0] if candidates else None


def load_table(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix == ".xlsx":
        return pd.read_excel(path)
    if suffix == ".jsonl":
        return pd.read_json(path, lines=True)
    if suffix == ".json":
        return pd.read_json(path)
    raise ValueError(f"Unsupported file type: {path}")


def normalize_dataset(df: pd.DataFrame) -> pd.DataFrame:
    input_candidates = ["input", "source", "sentence", "poem", "古诗", "原文", "text", "prompt"]
    ref_candidates = ["target", "reference", "answer", "modern", "translation", "现代文", "译文", "response", "output"]

    cols = list(df.columns)
    input_col = next((c for c in input_candidates if c in cols), None)
    ref_col = next((c for c in ref_candidates if c in cols), None)

    if input_col is None or ref_col is None:
        raise ValueError(
            f"Could not infer input/reference columns. Columns found: {cols}
"
            f"Please rename columns or edit normalize_dataset()."
        )

    out = df[[input_col, ref_col]].copy()
    out.columns = ["input", "reference"]
    out = out.dropna()
    out["input"] = out["input"].astype(str).str.strip()
    out["reference"] = out["reference"].astype(str).str.strip()
    out = out[(out["input"] != "") & (out["reference"] != "")]
    return out.reset_index(drop=True)


data_file = find_data_file()
if data_file:
    print("Loading data from:", data_file)
    raw_df = load_table(data_file)
    data = normalize_dataset(raw_df)
else:
    print("No data file found. Using toy examples for smoke test.")
    data = pd.DataFrame([
        {
            "input": "床前明月光，疑是地上霜。",
            "reference": "明亮的月光洒在床前，看起来像地上铺了一层霜。",
        },
        {
            "input": "举头望明月，低头思故乡。",
            "reference": "诗人抬头望着明月，又低下头思念自己的家乡。",
        },
        {
            "input": "春眠不觉晓，处处闻啼鸟。",
            "reference": "春夜睡得很沉，不知不觉天已亮，到处都能听见鸟叫声。",
        },
    ])

# Sample fixed benchmark split.
data = data.sample(frac=1.0, random_state=RANDOM_SEED).head(MAX_EXAMPLES).reset_index(drop=True)
print(data.shape)
display(data.head())

## 4. Prompt template

The prompt asks for a concise modern Chinese explanation and discourages hallucinated background information.

In [ ]:
def build_messages(poem_text: str) -> List[Dict[str, str]]:
    system = (
        "你是一位严谨的中国古典诗歌老师。"
        "任务是理解古诗句或诗段，并用现代汉语准确说明其意思。"
        "不要编造作者生平、历史背景或诗外信息。"
        "如果原文意象含蓄，要保留意象，不要过度解释。"
    )
    user = f"请将下面的古诗内容解释为自然、准确、简洁的现代汉语：

{poem_text}"
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


def make_prompt(tokenizer, poem_text: str) -> str:
    messages = build_messages(poem_text)
    if hasattr(tokenizer, "apply_chat_template"):
        try:
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            pass
    return messages[0]["content"] + "

用户：" + messages[1]["content"] + "

助手："

print(make_prompt(type("Dummy", (), {})(), data.loc[0, "input"]))

## 5. MLX helper functions

This cell loads one model at a time to keep memory pressure low. It records latency, generated token count, tokens/sec, and best-effort MLX memory stats.

In [ ]:
def clear_mlx_memory():
    gc.collect()
    try:
        mx.clear_cache()
    except Exception:
        pass
    try:
        mx.metal.clear_cache()
    except Exception:
        pass


def get_mlx_memory_gb() -> Dict[str, Optional[float]]:
    stats = {"active_memory_gb": None, "peak_memory_gb": None, "cache_memory_gb": None}
    try:
        stats["active_memory_gb"] = mx.metal.get_active_memory() / 1024**3
    except Exception:
        pass
    try:
        stats["peak_memory_gb"] = mx.metal.get_peak_memory() / 1024**3
    except Exception:
        pass
    try:
        stats["cache_memory_gb"] = mx.metal.get_cache_memory() / 1024**3
    except Exception:
        pass
    return stats


def reset_peak_memory():
    try:
        mx.metal.reset_peak_memory()
    except Exception:
        pass


def count_tokens(tokenizer, text: str) -> int:
    try:
        return len(tokenizer.encode(text))
    except Exception:
        try:
            return len(tokenizer(text)["input_ids"])
        except Exception:
            return max(1, len(text) // 2)


def generate_once(model, tokenizer, poem_text: str) -> Dict[str, Any]:
    prompt = make_prompt(tokenizer, poem_text)
    prompt_tokens = count_tokens(tokenizer, prompt)
    clear_mlx_memory()
    reset_peak_memory()

    t0 = time.perf_counter()
    first_token_latency = None
    output = ""

    # Prefer streaming when available so we can approximate first-token latency.
    if HAS_STREAM_GENERATE:
        try:
            chunks = []
            first = True
            kwargs = dict(
                model=model,
                tokenizer=tokenizer,
                prompt=prompt,
                max_tokens=MAX_TOKENS,
                temp=TEMPERATURE,
            )
            if MAX_KV_SIZE is not None:
                kwargs["max_kv_size"] = MAX_KV_SIZE

            for chunk in stream_generate(**kwargs):
                if first:
                    first_token_latency = time.perf_counter() - t0
                    first = False
                text = getattr(chunk, "text", None)
                if text is None:
                    text = str(chunk)
                chunks.append(text)
            output = "".join(chunks).strip()
        except Exception as e:
            # Fall back to non-streaming generation.
            output = None

    if output is None or output == "":
        kwargs = dict(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            max_tokens=MAX_TOKENS,
            verbose=False,
            temp=TEMPERATURE,
        )
        if MAX_KV_SIZE is not None:
            kwargs["max_kv_size"] = MAX_KV_SIZE
        output = generate(**kwargs).strip()

    total_latency = time.perf_counter() - t0
    generated_tokens = count_tokens(tokenizer, output)
    tok_per_sec = generated_tokens / total_latency if total_latency > 0 else None
    memory = get_mlx_memory_gb()

    return {
        "output": output,
        "prompt_tokens": prompt_tokens,
        "generated_tokens": generated_tokens,
        "latency_sec": total_latency,
        "first_token_latency_sec": first_token_latency,
        "tokens_per_sec": tok_per_sec,
        **memory,
    }

## 6. Run the benchmark

Run one model at a time. The model cache/download may take a while on first run.

If a 12B model fails due to memory pressure:

- reduce `MAX_TOKENS`
- set `MAX_KV_SIZE = 2048` or `4096`
- benchmark fewer examples
- use a smaller/4B/7B model for fine-tuning decisions

In [ ]:
all_rows = []

for model_label, model_id in MODELS.items():
    print("
" + "=" * 80)
    print(f"Loading {model_label}: {model_id}")
    print("=" * 80)

    clear_mlx_memory()
    reset_peak_memory()

    load_t0 = time.perf_counter()
    try:
        model, tokenizer = load(model_id, tokenizer_config={"trust_remote_code": True})
    except TypeError:
        # Older mlx-lm versions may not accept tokenizer_config.
        model, tokenizer = load(model_id)
    load_latency = time.perf_counter() - load_t0
    print(f"Loaded in {load_latency:.2f}s")
    print("Memory after load:", get_mlx_memory_gb())

    for idx, row in tqdm(data.iterrows(), total=len(data), desc=model_label):
        try:
            result = generate_once(model, tokenizer, row["input"])
            error = None
        except Exception as e:
            result = {
                "output": "",
                "prompt_tokens": None,
                "generated_tokens": None,
                "latency_sec": None,
                "first_token_latency_sec": None,
                "tokens_per_sec": None,
                "active_memory_gb": None,
                "peak_memory_gb": None,
                "cache_memory_gb": None,
            }
            error = repr(e)

        all_rows.append({
            "example_id": idx,
            "model_label": model_label,
            "model_id": model_id,
            "input": row["input"],
            "reference": row["reference"],
            "load_latency_sec": load_latency,
            "error": error,
            **result,
        })

    # Save after each model so partial progress is not lost.
    results_df = pd.DataFrame(all_rows)
    results_df.to_csv(OUT_DIR / "raw_generations.csv", index=False)

    del model, tokenizer
    clear_mlx_memory()

results_df = pd.DataFrame(all_rows)
display(results_df.head())
print("Saved:", OUT_DIR / "raw_generations.csv")

## 7. Automatic quality metrics

These are useful signals, not final truth.

For this task, **BERTScore + human review** is usually more meaningful than BLEU because modern Chinese restatements can be valid with different wording.

In [ ]:
# Keep metric imports here so generation still works if metric packages fail.
metric_df = results_df.copy()

# ROUGE-L
try:
    import evaluate
    rouge = evaluate.load("rouge")
    rouge_rows = []
    for (model_label), group in metric_df.groupby("model_label"):
        preds = group["output"].fillna("").tolist()
        refs = group["reference"].fillna("").tolist()
        scores = rouge.compute(predictions=preds, references=refs, use_stemmer=False)
        rouge_rows.append({"model_label": model_label, **scores})
    rouge_summary = pd.DataFrame(rouge_rows)
except Exception as e:
    print("ROUGE failed:", repr(e))
    rouge_summary = pd.DataFrame()

# SacreBLEU at corpus level; treat as secondary.
try:
    import sacrebleu
    bleu_rows = []
    for model_label, group in metric_df.groupby("model_label"):
        preds = group["output"].fillna("").tolist()
        refs = group["reference"].fillna("").tolist()
        score = sacrebleu.corpus_bleu(preds, [refs]).score
        bleu_rows.append({"model_label": model_label, "sacrebleu": score})
    bleu_summary = pd.DataFrame(bleu_rows)
except Exception as e:
    print("BLEU failed:", repr(e))
    bleu_summary = pd.DataFrame()

# BERTScore. For Chinese, this downloads a multilingual model on first run.
try:
    from bert_score import score as bert_score
    bert_rows = []
    for model_label, group in metric_df.groupby("model_label"):
        preds = group["output"].fillna("").tolist()
        refs = group["reference"].fillna("").tolist()
        P, R, F1 = bert_score(preds, refs, lang="zh", verbose=True)
        bert_rows.append({
            "model_label": model_label,
            "bertscore_precision": float(P.mean()),
            "bertscore_recall": float(R.mean()),
            "bertscore_f1": float(F1.mean()),
        })
    bert_summary = pd.DataFrame(bert_rows)
except Exception as e:
    print("BERTScore failed:", repr(e))
    bert_summary = pd.DataFrame()

# Merge summaries.
summary = metric_df.groupby("model_label").agg(
    n=("example_id", "count"),
    error_count=("error", lambda s: s.notna().sum()),
    avg_latency_sec=("latency_sec", "mean"),
    median_latency_sec=("latency_sec", "median"),
    avg_first_token_latency_sec=("first_token_latency_sec", "mean"),
    avg_tokens_per_sec=("tokens_per_sec", "mean"),
    avg_generated_tokens=("generated_tokens", "mean"),
    peak_memory_gb=("peak_memory_gb", "max"),
).reset_index()

for extra in [rouge_summary, bleu_summary, bert_summary]:
    if not extra.empty:
        summary = summary.merge(extra, on="model_label", how="left")

summary.to_csv(OUT_DIR / "summary_metrics.csv", index=False)
display(summary)
print("Saved:", OUT_DIR / "summary_metrics.csv")

## 8. Speed and memory charts

In [ ]:
if not summary.empty:
    for col, title in [
        ("avg_latency_sec", "Average latency per example, lower is better"),
        ("avg_tokens_per_sec", "Average generation speed, higher is better"),
        ("peak_memory_gb", "Peak MLX memory GB, lower is better"),
    ]:
        if col in summary.columns and summary[col].notna().any():
            plt.figure(figsize=(8, 4))
            plt.bar(summary["model_label"], summary[col])
            plt.title(title)
            plt.ylabel(col)
            plt.xticks(rotation=30, ha="right")
            plt.tight_layout()
            plt.show()

## 9. Side-by-side qualitative comparison

This table is often the most valuable part for poem understanding.

In [ ]:
wide = results_df.pivot_table(
    index=["example_id", "input", "reference"],
    columns="model_label",
    values="output",
    aggfunc="first",
).reset_index()

wide.to_csv(OUT_DIR / "side_by_side_outputs.csv", index=False)
display(wide.head(10))
print("Saved:", OUT_DIR / "side_by_side_outputs.csv")

## 10. Human evaluation sheet

Use this for manual scoring. Suggested rubric:

- `accuracy_1_5`: Does it preserve the correct meaning?
- `fluency_1_5`: Is the modern Chinese natural?
- `imagery_1_5`: Does it preserve poetic imagery without flattening it too much?
- `hallucination_1_5`: 5 means no hallucination; 1 means serious invented content.
- `overall_1_5`: Your final preference.

In [ ]:
human_eval = results_df[["example_id", "model_label", "input", "reference", "output"]].copy()
for col in ["accuracy_1_5", "fluency_1_5", "imagery_1_5", "hallucination_1_5", "overall_1_5", "notes"]:
    human_eval[col] = ""

human_eval_path_csv = OUT_DIR / "human_eval_template.csv"
human_eval_path_xlsx = OUT_DIR / "human_eval_template.xlsx"
human_eval.to_csv(human_eval_path_csv, index=False)
human_eval.to_excel(human_eval_path_xlsx, index=False)

print("Saved:", human_eval_path_csv)
print("Saved:", human_eval_path_xlsx)
display(human_eval.head())

## 11. Decision guide

After running this benchmark, use this decision rule:

1. If **Qwen is clearly better in quality and faster**, fine-tune Qwen first.
2. If **Gemma 12B is better but much slower**, consider using Gemma only for teacher-labeling or final evaluation.
3. If **Gemma 12B quality is only slightly better**, choose Qwen for local LoRA fine-tuning because iteration speed matters.
4. If both are weak, improve the prompt and examples before fine-tuning.

For local fine-tuning on a 32GB M4 Mac mini, prioritize:

- Qwen 7B/8B LoRA or QLoRA
- Gemma 4B/E4B LoRA
- 12B adapter tuning only after you confirm the smaller setup works